# Módulo 3: Agentes de IA
- **Sesión:** 01 — Fundamentos de agentes de IA
- **Notebook:** Memoria del agente
- **Fecha:** 20/10/2026

In [2]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
import os

In [3]:

# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

# Configuración del modelo de lenguaje
load_dotenv(override = True)

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)



In [4]:
PEDIDOS = {
    "48213": "en ruta, llega el 26/10/2026 a Arequipa",
    "50555": "en preparación, llega el 30/10/2026 a Cusco",
}


@tool
def consultar_pedido(numero_pedido: str) -> str:
    """Consulta el estado y la fecha de llegada de un pedido de TiendaYa."""
    return PEDIDOS.get(numero_pedido, f"No existe el pedido {numero_pedido}.")


SYSTEM_PROMPT = ("Eres el asistente de TiendaYa. Usa las herramientas para obtener los datos; "
                 "no los inventes. Responde de forma breve.")


def preguntar(agente, texto, config=None):
    """Envía un mensaje al agente y devuelve el resultado completo."""
    return agente.invoke({"messages": [{"role": "user", "content": texto}]}, config=config)


print(consultar_pedido.invoke({"numero_pedido": "48213"}))


en ruta, llega el 26/10/2026 a Arequipa


## Paso 1 — Un agente sin memoria


In [9]:
from langchain.agents import create_agent

agente = create_agent(
    model=llm,
    tools=[consultar_pedido],
    system_prompt=SYSTEM_PROMPT
)

resp1 = preguntar(agente, "Dónde esta mi pedido 48213?")
print(resp1['messages'][-1].text)

resp2 = preguntar(agente, "Y cuando llega?")
print(resp2['messages'][-1].text)



Tu pedido 48213 está **en ruta** y llegará el **26/10/2026** a Arequipa.
Para poder decirte cuándo llega, necesito tu **número de pedido**. ¿Podrías proporcionármelo, por favor?


## Paso 2 — Un agente con memoria


In [15]:
from langchain.agents import create_agent
# No se necesita instalar la libreria de langgraph, esta contenida en el core de langchain
from langgraph.checkpoint.memory import InMemorySaver 


memoria = InMemorySaver() 

agente = create_agent(   # graph
    model=llm,
    tools=[consultar_pedido],
    system_prompt=SYSTEM_PROMPT,
    checkpointer= memoria # Guarda la informacion en memoria RAM
)


config1 = { "configurable" : 
            {"thread_id" : "user01"}
        }

config2 = { "configurable" : 
            {"thread_id" : "user02"}
        }


resp1 = preguntar(agente, "Dónde esta mi pedido 48213?", config1)
print("user01 :" , resp1['messages'][-1].text)

resp3 = preguntar(agente, "Dónde esta mi pedido 50555?", config2)
print("user02 :" , resp3['messages'][-1].text)


resp2 = preguntar(agente, "Y cuando llega?", config1)
print("user01 :" , resp2['messages'][-1].text)

resp4 = preguntar(agente, "Y cuando llega?", config2)
print("user02 :" , resp4['messages'][-1].text)

memoria.delete_thread( "user01")

resp5 = preguntar(agente, "Y cuando llega?", config1)
print("user01 :" , resp5['messages'][-1].text)


user01 : Tu pedido 48213 está en ruta y llegará el 26/10/2026 a Arequipa.
user02 : Tu pedido 50555 está **en preparación** y llegará el **30/10/2026** a Cusco.
user01 : El pedido llegará el **26/10/2026**.
user02 : Llega el **30/10/2026**.
user01 : Para poder consultar cuándo llega tu pedido, por favor dime tu **número de pedido**.
